# CSCI316 Project 2 — PyTorch Specific: Text Classification
**Task:** Sentiment Analysis on Tamil-English Code-Switched Text

This notebook mirrors the structure of `main.ipynb` (up to the pre-PEFT section) but uses a pure PyTorch pipeline with no Hugging Face Transformers.

## Cell 1 — Install Dependencies

In [ ]:
# Run this first
!pip install torch pandas scikit-learn matplotlib seaborn --quiet

## Cell 2 — Imports & Reproducibility

In [ ]:
import os
import re
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Cell 3 — Configuration

In [ ]:
CONFIG = {
    "train_path": "tamilmix_train.csv",
    "val_path": "tamilmix_val.csv",
    "test_path": "tamilmix_test.csv",
    "num_labels": 5,
    "max_length": 64,
    "min_freq": 2,
    "embed_dim": 256,
    "hidden_dim": 256,
    "dropout": 0.3,
    "batch_size": 64,
    "num_epochs": 8,
    "learning_rate": 2e-3,
    "weight_decay": 1e-4,
    "warmup_ratio": 0.1,
    "grad_clip": 1.0,
    "use_class_weights": True,
    "output_dir": "pytorch_model",
    "results_dir": "results_pytorch_only",
}

LABEL_NAMES = ["Positive", "Negative", "Mixed_feelings", "unknown_state", "not-Tamil"]
os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["results_dir"], exist_ok=True)
print("Config loaded.")

## Cell 4 — Load Cleaned Data

In [ ]:
train_df = pd.read_csv(CONFIG["train_path"])
val_df = pd.read_csv(CONFIG["val_path"])
test_df = pd.read_csv(CONFIG["test_path"])

for df in [train_df, val_df, test_df]:
    df["text_clean"] = df["text_clean"].fillna("").astype(str)
    df["label_int"] = df["label"].astype(int)

print(f"Train:      {len(train_df)} rows")
print(f"Validation: {len(val_df)} rows")
print(f"Test:       {len(test_df)} rows")
print("\nLabel distribution (train):")
print(train_df["label_int"].value_counts().sort_index())

## Cell 5 — Compute Class Weights

In [ ]:
train_labels = train_df["label_int"].values
present_classes = np.unique(train_labels)
computed_weights = compute_class_weight(
    class_weight="balanced",
    classes=present_classes,
    y=train_labels,
)

class_weights = np.ones(CONFIG["num_labels"], dtype=np.float32)
for cls, w in zip(present_classes, computed_weights):
    class_weights[cls] = w

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print("Class weights:")
for i, (name, w) in enumerate(zip(LABEL_NAMES, class_weights)):
    print(f"[{i}] {name:<20} {w:.4f}")

## Cell 6 — Tokenization (PyTorch-Only Vocabulary)

In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

def basic_tokenize(text: str):
    return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)

def build_vocab(texts, min_freq=2):
    counter = Counter()
    for t in texts:
        counter.update(basic_tokenize(t))

    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for tok, freq in counter.items():
        if freq >= min_freq and tok not in vocab:
            vocab[tok] = len(vocab)
    return vocab

def encode_text(text, vocab, max_length):
    tokens = basic_tokenize(text)
    ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in tokens][:max_length]
    attn = [1] * len(ids)
    while len(ids) < max_length:
        ids.append(vocab[PAD_TOKEN])
        attn.append(0)
    return ids, attn

vocab = build_vocab(train_df["text_clean"].tolist(), min_freq=CONFIG["min_freq"])
print(f"Vocab size: {len(vocab):,}")
sample = "thala vera level . luv u so much"
sample_ids, _ = encode_text(sample, vocab, CONFIG["max_length"])
print("Sample:", sample)
print("Encoded IDs (first 20):", sample_ids[:20])

## Cell 7 — PyTorch Dataset Class

In [ ]:
class TamilSentimentDataset(Dataset):
    def __init__(self, dataframe, vocab, max_length):
        self.texts = dataframe["text_clean"].tolist()
        self.labels = dataframe["label_int"].tolist()
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        input_ids, attention_mask = encode_text(self.texts[idx], self.vocab, self.max_length)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = TamilSentimentDataset(train_df, vocab, CONFIG["max_length"])
val_dataset = TamilSentimentDataset(val_df, vocab, CONFIG["max_length"])
test_dataset = TamilSentimentDataset(test_df, vocab, CONFIG["max_length"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## Cell 8 — Model Initialisation (Full Fine-Tuning in PyTorch)

In [ ]:
class BiLSTMSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, pad_idx=0, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.encoder = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids, attention_mask):
        emb = self.embedding(input_ids)
        enc_out, _ = self.encoder(emb)

        mask = attention_mask.unsqueeze(-1).float()
        masked = enc_out * mask
        pooled = masked.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-8)

        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits

model = BiLSTMSentimentClassifier(
    vocab_size=len(vocab),
    embed_dim=CONFIG["embed_dim"],
    hidden_dim=CONFIG["hidden_dim"],
    num_labels=CONFIG["num_labels"],
    pad_idx=vocab[PAD_TOKEN],
    dropout=CONFIG["dropout"],
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Cell 9 — Optimizer, Scheduler & Loss

In [ ]:
optimizer = AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])

total_steps = len(train_loader) * CONFIG["num_epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])

scheduler = OneCycleLR(
    optimizer,
    max_lr=CONFIG["learning_rate"],
    total_steps=total_steps,
    pct_start=CONFIG["warmup_ratio"],
)

loss_fn = nn.CrossEntropyLoss(
    weight=class_weights_tensor if CONFIG["use_class_weights"] else None
)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Using class weights: {CONFIG['use_class_weights']}")

## Cell 10 — Training & Evaluation Functions

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, loss_fn, device, grad_clip):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for step, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=-1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.detach().cpu().numpy())

        if (step + 1) % 100 == 0:
            print(f"    Step {step+1}/{len(loader)} - loss: {loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    return avg_loss, acc, f1, all_preds, all_labels

## Cell 11 — Training Loop

In [ ]:
history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "val_f1": []
}

best_val_f1 = -1.0
best_epoch = -1
best_model_path = os.path.join(CONFIG["output_dir"], "best_model.pt")

print("=" * 60)
print("TRAINING — PyTorch Only")
print("=" * 60)

for epoch in range(CONFIG["num_epochs"]):
    print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
    print("-" * 40)

    tr_loss, tr_acc = train_epoch(
        model, train_loader, optimizer, scheduler, loss_fn, DEVICE, CONFIG["grad_clip"]
    )
    va_loss, va_acc, va_f1, _, _ = evaluate(model, val_loader, loss_fn, DEVICE)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)
    history["val_f1"].append(va_f1)

    print(f"  Train - Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}")
    print(f"  Val   - Loss: {va_loss:.4f}  Acc: {va_acc:.4f}  F1: {va_f1:.4f}")

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        best_epoch = epoch + 1
        torch.save(model.state_dict(), best_model_path)
        print(f"  Saved best model (val F1 = {best_val_f1:.4f})")

print(f"\nBest epoch: {best_epoch} | Best val F1: {best_val_f1:.4f}")

## Cell 12 — Plot Training Curves

In [ ]:
epochs = np.arange(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, history["train_loss"], label="Train")
plt.plot(epochs, history["val_loss"], label="Val")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, history["train_acc"], label="Train")
plt.plot(epochs, history["val_acc"], label="Val")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs, history["val_f1"], label="Val F1")
plt.title("Weighted F1")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## Cell 13 — Load Best Checkpoint

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print(f"Loaded: {best_model_path}")

## Cell 14 — Final Test Evaluation

In [ ]:
test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(model, test_loader, loss_fn, DEVICE)

print("=" * 60)
print("TEST RESULTS")
print("=" * 60)
print(f"Loss:         {test_loss:.4f}")
print(f"Accuracy:     {test_acc:.4f}")
print(f"Weighted F1:  {test_f1:.4f}")

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=LABEL_NAMES, digits=4, zero_division=0))

## Cell 15 — Confusion Matrix

In [ ]:
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title("Confusion Matrix (Test)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

## Cell 16 — Qualitative Examples

In [ ]:
qual_df = test_df[["text_clean", "label_int"]].copy()
qual_df["pred_int"] = test_preds
qual_df["true_label"] = qual_df["label_int"].map(lambda x: LABEL_NAMES[int(x)])
qual_df["pred_label"] = qual_df["pred_int"].map(lambda x: LABEL_NAMES[int(x)])
qual_df["correct"] = qual_df["label_int"] == qual_df["pred_int"]

print("Correct examples:")
display(qual_df[qual_df["correct"]].head(5)[["text_clean", "true_label", "pred_label"]])

print("\nIncorrect examples:")
display(qual_df[~qual_df["correct"]].head(5)[["text_clean", "true_label", "pred_label"]])

qual_out = os.path.join(CONFIG["results_dir"], "qualitative_examples.csv")
qual_df.to_csv(qual_out, index=False)
print(f"Saved qualitative file: {qual_out}")

## Cell 17 — Inference Helper

In [ ]:
def predict_text(text: str):
    model.eval()
    ids, mask = encode_text(text, vocab, CONFIG["max_length"])
    input_ids = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    attention_mask = torch.tensor([mask], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
        pred_id = int(np.argmax(probs))

    return {
        "label_id": pred_id,
        "label_name": LABEL_NAMES[pred_id],
        "probabilities": {LABEL_NAMES[i]: float(probs[i]) for i in range(len(LABEL_NAMES))},
    }

print(predict_text("This movie was super, but konjam long-a irundhudhu"))

## Cell 18 — Save Artifacts (PyTorch-Only)

In [ ]:
torch.save(model.state_dict(), os.path.join(CONFIG["output_dir"], "final_model.pt"))

with open(os.path.join(CONFIG["output_dir"], "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)

metrics_df = pd.DataFrame(
    {
        "best_epoch": [best_epoch],
        "best_val_f1": [best_val_f1],
        "test_loss": [test_loss],
        "test_acc": [test_acc],
        "test_f1": [test_f1],
    }
)
metrics_path = os.path.join(CONFIG["results_dir"], "metrics_pytorch_only.csv")
metrics_df.to_csv(metrics_path, index=False)

print("Saved model + vocab + metrics.")
print(f"Model dir:   {CONFIG['output_dir']}")
print(f"Results dir: {CONFIG['results_dir']}")